# Final Research Pipeline Colab
Notebook final untuk eksekusi pipeline penelitian dari Step 1 sampai Step 11 menggunakan `data/dataset_final.csv`.

## 0) Setup Runtime
Pilih GPU T4 di Colab: Runtime > Change runtime type > T4 GPU.

In [ ]:
import os
HF_TOKEN = 'hf_xxx_your_token_here'  # placeholder
os.environ['HF_TOKEN'] = HF_TOKEN

!git clone https://github.com/kzquandary/mbg-sentiment.git
%cd mbg-sentiment
!git checkout main
!python3 -m pip install -r requirements.txt
!nvidia-smi

## 1) Konfigurasi Final
Model config final: `src/resources/step7_final_production.json`
Trial: `final_best_basep1_finetune1_len96_bs4_h128_ce`

In [ ]:
import os
assert os.path.exists('data/dataset_final.csv'), 'Dataset final tidak ditemukan di repo'
assert os.path.exists('src/resources/step7_final_production.json'), 'Config final tidak ditemukan'
print('Dataset:', 'data/dataset_final.csv')
print('Config :', 'src/resources/step7_final_production.json')

## 2) Run Pipeline Step 1-11 (End-to-End)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:64'
!python3 run_pipeline_full.py \
  --from-step 1 \
  --until-step 11 \
  --dataset-source final \
  --step7-config-json src/resources/step7_final_production.json \
  --step7-max-trials 1 \
  --run-class-multiplier

## 3) Output Inline Per Step
Cell ini menampilkan ringkasan hasil setiap step langsung di notebook (table/figure), bukan hanya file artifact.


In [ ]:
import os
import json
import pandas as pd
from IPython.display import display, Markdown, Image


def show_title(text):
    display(Markdown(f"## {text}"))


def show_subtitle(text):
    display(Markdown(f"### {text}"))


def show_csv(path, n=10):
    if os.path.exists(path):
        df = pd.read_csv(path)
        display(df.head(n))
        return df
    print(f"[SKIP] File tidak ditemukan: {path}")
    return None


def show_json_flat(path, n_cols=30):
    if not os.path.exists(path):
        print(f"[SKIP] File tidak ditemukan: {path}")
        return None
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    flat = pd.json_normalize(data, sep='.')
    if flat.shape[1] > n_cols:
        flat = flat.iloc[:, :n_cols]
    display(flat.T.rename(columns={0: 'value'}))
    return data


def show_markdown_file(path, max_chars=2500):
    if not os.path.exists(path):
        print(f"[SKIP] File tidak ditemukan: {path}")
        return
    with open(path, encoding='utf-8') as f:
        text = f.read()
    display(Markdown(text[:max_chars]))


def show_image(path, width=900):
    if os.path.exists(path):
        display(Image(filename=path, width=width))
    else:
        print(f"[SKIP] File tidak ditemukan: {path}")


show_title('Step 1 - Audit Data')
audit = show_json_flat('outputs/data_audit_summary.json')
if audit and 'audit' in audit:
    core = audit['audit']
    rows = [{
        'n_rows': core.get('n_rows'),
        'n_cols': core.get('n_cols'),
        'text_col_detected': core.get('text_col_detected'),
        'label_col_detected': core.get('label_col_detected'),
        'duplicate_full_rows': core.get('duplicate_full_rows'),
        'duplicate_text_rows': core.get('duplicate_text_rows'),
    }]
    show_subtitle('Ringkasan Audit')
    display(pd.DataFrame(rows))
    if core.get('label_distribution_raw'):
        show_subtitle('Distribusi Label (Raw)')
        label_df = pd.DataFrame(
            [{'label': k, 'count': v} for k, v in core['label_distribution_raw'].items()]
        ).sort_values('count', ascending=False)
        display(label_df)
show_subtitle('Preview Data')
show_csv('outputs/data_audit_preview.csv', n=10)


show_title('Step 2 - Cleaning')
show_json_flat('outputs/cleaning_log.json')
show_subtitle('Dataset Setelah Cleaning')
show_csv('data/dataset_pipeline_clean.csv', n=10)


show_title('Step 3 - Split + Preprocess')
split_summary = show_json_flat('outputs/split_summary.json')
if split_summary and 'size' in split_summary:
    show_subtitle('Ukuran Split')
    display(pd.DataFrame([split_summary['size']]))
show_subtitle('Sample Hasil Preprocess')
show_csv('outputs/preprocessing_samples.csv', n=10)


show_title('Step 4 - EDA')
show_subtitle('EDA Summary (excerpt)')
show_markdown_file('outputs/eda_summary.md', max_chars=2500)
show_subtitle('EDA Figures')
for fig in [
    'outputs/figures/label_distribution.png',
    'outputs/figures/text_length_boxplot.png',
    'outputs/figures/text_length_histogram.png',
    'outputs/figures/top_words_general.png',
    'outputs/figures/wordcloud_general.png',
]:
    show_image(fig, width=820)


show_title('Step 5 - Split Artifact Check')
for split_file in ['data/train.csv', 'data/test.csv']:
    if os.path.exists(split_file):
        sdf = pd.read_csv(split_file)
        info = pd.DataFrame([{
            'file': split_file,
            'rows': len(sdf),
            'cols': sdf.shape[1],
        }])
        display(info)


show_title('Step 6 - Baseline Models')
show_subtitle('Baseline Results (val-best + final test)')
show_csv('outputs/baseline_results.csv', n=20)
show_subtitle('Top Validation Trials per Baseline')
if os.path.exists('outputs/baseline_trials.csv'):
    bt = pd.read_csv('outputs/baseline_trials.csv')
    top_trials = bt.sort_values(['model', 'f1_macro'], ascending=[True, False]).groupby('model').head(5)
    display(top_trials)
else:
    print('[SKIP] File tidak ditemukan: outputs/baseline_trials.csv')
show_subtitle('Baseline Comparison (excerpt)')
show_markdown_file('outputs/baseline_comparison.md', max_chars=2000)


show_title('Step 7 - IndoBERT + BiLSTM')
show_subtitle('Best Config')
show_json_flat('outputs/step7_best_config.json')
show_subtitle('Trial Summary')
show_csv('outputs/step7_trials.csv', n=20)
show_subtitle('Training History (last rows)')
if os.path.exists('outputs/training_history.csv'):
    hist = pd.read_csv('outputs/training_history.csv')
    display(hist.tail(20))
else:
    print('[SKIP] File tidak ditemukan: outputs/training_history.csv')
show_subtitle('Model Architecture (excerpt)')
show_markdown_file('outputs/model_architecture.md', max_chars=2000)


show_title('Step 8 - Tuning (Opsional)')
show_csv('outputs/experiment_results.csv', n=20)
show_json_flat('outputs/best_config.json')


show_title('Step 9 - Final Evaluation (Held-out Test)')
metrics = show_json_flat('outputs/final_metrics.json')
if metrics:
    metric_cols = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
    display(pd.DataFrame([metrics])[metric_cols])
show_subtitle('Classification Report')
show_csv('outputs/classification_report.csv', n=20)
show_subtitle('Confusion Matrix (CSV)')
if os.path.exists('outputs/confusion_matrix.csv'):
    cm = pd.read_csv('outputs/confusion_matrix.csv', index_col=0)
    display(cm)
else:
    print('[SKIP] File tidak ditemukan: outputs/confusion_matrix.csv')
show_subtitle('Confusion Matrix (Image)')
show_image('outputs/figures/confusion_matrix.png', width=760)


show_title('Step 10 - Error Analysis')
show_subtitle('Error Analysis Summary (excerpt)')
show_markdown_file('outputs/error_analysis_summary.md', max_chars=2500)
if os.path.exists('outputs/error_analysis_samples.xlsx'):
    try:
        ea = pd.read_excel('outputs/error_analysis_samples.xlsx')
        show_subtitle('Error Samples')
        display(ea.head(20))
    except Exception as e:
        print(f"[SKIP] Gagal baca outputs/error_analysis_samples.xlsx: {e}")


show_title('Step 11 - Draft Laporan')
show_subtitle('Bab 4 (excerpt)')
show_markdown_file('outputs/bab4_hasil_otomatis.md', max_chars=3000)
show_subtitle('Bab 5 (excerpt)')
show_markdown_file('outputs/bab5_kesimpulan_saran_otomatis.md', max_chars=3000)


## 4) Artifact Final (Tetap Tersimpan di File)
- `outputs/pipeline_run_log.json`
- `outputs/final_metrics.json`
- `outputs/classification_report.csv`
- `outputs/confusion_matrix.csv`
- `outputs/error_analysis_summary.md`
- `outputs/bab4_hasil_otomatis.md`
- `outputs/bab5_kesimpulan_saran_otomatis.md`
